# Portion 1 · IBM AML HI-Small Data Pipeline

Read the source transactions and laundering patterns. Build time-based splits, account sequences, training-period vocabularies, historical features, and stratified label subsets. Save auditable outputs under the repository's Portion1 folder.

Adapt the bank/account key and temporal-ordering ideas from [IBM Multi-GNN's preprocessing script](https://github.com/IBM/Multi-GNN/blob/main/format_kaggle_files.py). Preserve source transaction fields and build separate sequence inputs. The graph-model formatter is a reference and is not run directly.


## 1. Locate the GitHub checkout

`REPO_URL` is already set to the project repository in the first code cell. When this step runs in Colab, it clones the repository and retrieves the dataset files through Git LFS. The temporary SQLite database stays on Colab's local disk.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ted-cc6/TabBERT_AML"  # Set to the project's HTTPS GitHub clone URL in Colab.
WORK_DB = Path("/content/aml_portion1_work.sqlite")
current = Path.cwd()
PROJECT_ROOT = next(
    (path for path in (current, current / "Portion1")
     if (path / "portion1_pipeline.py").is_file()),
    None,
)

if PROJECT_ROOT is None:
    clone_root = Path("/content/aml_project_repo")
    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)
    if not clone_root.exists():
        if not REPO_URL:
            raise ValueError("Set REPO_URL to the project GitHub clone URL.")
        subprocess.run(["git", "clone", REPO_URL, str(clone_root)], check=True)
    subprocess.run(["git", "-C", str(clone_root), "lfs", "pull"], check=True)
    PROJECT_ROOT = next(
        (path for path in (clone_root / "Portion1", clone_root)
         if (path / "portion1_pipeline.py").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("portion1_pipeline.py was not found in the checkout.")

sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


## 2. Check the source files

Read the Kaggle IBM AML HI-Small transactions and patterns. The transaction file is gzip-compressed without changing the source CSV. Check file sizes and the header. Download only these two files if they are missing; do not fetch the full dataset.

Source: [IBM Transactions for Anti Money Laundering (AML)](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml). A Kaggle login may be required. Keep API credentials out of this notebook.


In [ ]:
import csv
import gzip
import shutil
import subprocess

raw_dir = PROJECT_ROOT / "raw"
raw_dir.mkdir(exist_ok=True)
transactions_gz = raw_dir / "HI-Small_Trans.csv.gz"
patterns_txt = raw_dir / "HI-Small_Patterns.txt"

if not transactions_gz.exists() or not patterns_txt.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub
    handle = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
    if not transactions_gz.exists():
        source = Path(kagglehub.dataset_download(handle, path="HI-Small_Trans.csv"))
        with source.open("rb") as incoming, gzip.open(transactions_gz, "wb", compresslevel=9) as outgoing:
            shutil.copyfileobj(incoming, outgoing)
    if not patterns_txt.exists():
        source = Path(kagglehub.dataset_download(handle, path="HI-Small_Patterns.txt"))
        shutil.copy2(source, patterns_txt)

with gzip.open(transactions_gz, "rt", encoding="utf-8-sig", newline="") as stream:
    header = next(csv.reader(stream))
print("Transactions:", transactions_gz.name, transactions_gz.stat().st_size, "bytes")
print("Patterns:", patterns_txt.name, patterns_txt.stat().st_size, "bytes")
print("Columns:", header)


## 3. Ingest transactions and match patterns

Assign transaction_id from the CSV data-row number. Combine bank and account IDs into an account key. Match the eight laundering typologies against complete pattern-file transaction rows. Expand each transaction into sender and receiver account events. Stop on missing fields, invalid amounts, or unmatched pattern-file rows.


In [ ]:
from portion1_pipeline import connect, ingest

con = connect(WORK_DB)
try:
    ingest(PROJECT_ROOT, con)
finally:
    con.close()
state = json.loads((PROJECT_ROOT / "processed/pipeline_state.json").read_text())
print({key: state[key] for key in ("transaction_count", "positive_count", "event_count", "matched_pattern_row_keys")})


## 4. Inspect distributions and set time splits

Compute daily volume, laundering rates, typology counts, and account-sequence lengths. Split the main September 1–10 period by elapsed time into 60% train, 20% valid, and 20% test. Keep September 11 onward as a separate tail audit set. Select a window length from 16, 32, and 64 using the training-period 90th percentile.


In [ ]:
from portion1_pipeline import eda

con = connect(WORK_DB)
try:
    eda(PROJECT_ROOT, con)
finally:
    con.close()
summary = json.loads((PROJECT_ROOT / "reports/eda_summary.json").read_text())
print("Split counts:", summary["split_counts"])
print("Sequence lengths:", summary["account_event_length"])
print("Selected window:", summary["recommended_sequence_length"])
print("Tail:", summary["after_2022_09_10"])


## 5. View EDA charts

Compare daily transaction counts, daily laundering counts and rates, account-event sequence lengths, and typology counts. Read the date labels for the full time span and the account chart title for the recommended 32-event window.


In [ ]:
from IPython.display import SVG, display

for name in (
    "daily_transactions.svg",
    "daily_laundering.svg",
    "daily_laundering_rate.svg",
    "account_sequence_lengths.svg",
    "typology_counts.svg",
):
    display(SVG(filename=str(PROJECT_ROOT / "figures" / name)))


## 6. Build tokens, windows, and baseline features

Fit category vocabularies, amount bins, and time-gap bins on train only. Order events by account, time, and transaction ID. Use the sender event as each transaction's single prediction target. Export six-field tokens and causal window indices for the encoder. Export 1-hour, 24-hour, and 7-day history features for baselines. Aggregate amounts only within the same currency.


In [ ]:
from portion1_pipeline import export

con = connect(WORK_DB)
try:
    export(PROJECT_ROOT, con)
finally:
    con.close()
print("Token archive:", (PROJECT_ROOT / "processed/event_tokens.npz").stat().st_size, "bytes")
print("Data contract:", PROJECT_ROOT / "DATA_CONTRACT.md")


## 7. Create stratified label subsets

Sample labels only from train, shuffling positives and negatives separately. Use seeds 42, 43, and 44 to create nested 1%, 5%, and 20% subsets; add 2% when the 1% subset has fewer than 30 positives. Export the complete 100% train list once in `processed/label_subset_100pct.csv.gz` and reuse it for every seed.


In [ ]:
from portion1_pipeline import subsets

con = connect(WORK_DB)
try:
    subsets(PROJECT_ROOT, con)
finally:
    con.close()
manifest = json.loads((PROJECT_ROOT / "processed/label_subset_manifest.json").read_text())
print(manifest)
print("100% train list:", PROJECT_ROOT / "processed/label_subset_100pct.csv.gz")


## 8. Verify outputs and hand off data

Check token shape, target-window bounds, split totals, typology matches, and the complete 100% train list. Read `DATA_CONTRACT.md` for field order, prediction unit, time boundaries, and leakage guards.


In [ ]:
from portion1_pipeline import verify
import numpy as np

con = connect(WORK_DB)
try:
    verify(PROJECT_ROOT, con)
finally:
    con.close()
tokens = np.load(PROJECT_ROOT / "processed/event_tokens.npz")["tokens"]
print("Token shape:", tokens.shape)
verification = json.loads((PROJECT_ROOT / "reports/verification.json").read_text())
print("Verification status:", verification["status"])
print("100% train list:", verification["full_train_list_rows"], "rows,", verification["full_train_list_positives"], "positives")
